# 32.03 Набор размеров при неизвестной $h$

> **Статус:** канонический синтетический анализ будущей сокращённой постановки
> без индивидуальной КТ как обязательного входа. Он не доказывает, что $h$
> можно надёжно оценить у пациента, и не выбирает число сборок для клиники.

Notebook не читает данные добровольцев или пациентов. Его задача — проверить
структурный ранг и показать, какая дополнительная информация нужна для перехода
от полной разработочной постановки `32.02` к сокращённой.


## Место в стратегии разработки методики

На добровольцах $h$ сначала должна быть получена/проверена по КТ и использована
в полной постановке. Только после независимой проверки сокращённая задача может
оценивать эффективную $h$ одновременно с $\rho_1,\rho_2$. Расчётная $h$ остаётся
параметром двуслойной модели и не называется анатомической толщиной до сравнения
с КТ.

Этот notebook относится к будущей целевой методике, а не к уже валидированному
пациентскому расчёту. МГТУ и РНЦХ должны получать отдельные калибровки и модели
ошибок. РЕО32 эксперимента 1 сюда не входит.


## Структурные условия

При калиброванном тракте, одном статическом состоянии и неизвестных

$$\boldsymbol\vartheta=(\ln\rho_1,\ln\rho_2,\ln h)$$

используется матрица

$$
\widetilde J_K=\left[
S_{\rho_1}(L_k),S_{\rho_2}(L_k),S_h(L_k)
\right]_{k\in K}.
$$

- Два размера не могут дать ранг 3: минимально нужны три разных наблюдения.
- Три и более размера могут иметь локальный ранг 3 в идеальной модели, но
  машинный ранг не означает практическую определимость.
- При неизвестном multiplicative gain добавляется столбец единиц. Из
  $S_{\rho_1}+S_{\rho_2}=1$ следует точная зависимость столбцов и абсолютный
  масштаб сопротивлений остаётся неидентифицируемым при любом числе размеров.
- Общие $h$ и $\rho_1$ между вдохом/выдохом могли бы добавить наблюдения, но это
  два отдельных физиологических допущения. Они не вводятся скрыто.

Ниже применяется только знаковый вещественный $Z$ и единичное относительное
взвешивание. Реальный модуль/фаза, ковариация, fixed-order и ошибка модели пока
не включены.


## Почему «взять все десять» не является решением задачи выбора

При независимых одинаковых ошибках и нулевой цене измерения добавление строки
увеличивает информационную матрицу на положительно полуопределённое слагаемое.
Поэтому формальный критерий информации тривиально предпочитает все доступные
размеры. Чтобы найти **минимальный** набор, заранее нужны:

1. допустимая неопределённость целевых $h$ и $\rho_2$;
2. стоимость/длительность дополнительной последовательной записи;
3. ковариация и временная неповторяемость;
4. допустимое субъектное множество по $L_{max}$;
5. правило holdout-проверки и отказа при плоском профиле.

Без этих входов можно сравнить лучший идеализированный набор каждой мощности,
но нельзя оптимизировать число измерений или назначить клинический протокол.


In [1]:
from itertools import combinations
from pathlib import Path
import platform
import sys

import numpy as np

candidates = [Path.cwd(), Path.cwd() / "Colab Notebooks", Path.cwd().parent]
model_paths = sorted({
    (candidate / "two_layer_model.py").resolve()
    for candidate in candidates
    if (candidate / "two_layer_model.py").is_file()
})
if len(model_paths) != 1:
    raise RuntimeError(f"expected one canonical two_layer_model.py, found: {model_paths}")

model_path = model_paths[0]
sys.path.insert(0, str(model_path.parent))
import two_layer_model as tlm

if Path(tlm.__file__).resolve() != model_path:
    raise RuntimeError(f"imported non-canonical model: {tlm.__file__}")

print(f"Python {platform.python_version()}; NumPy {np.__version__}")
print(f"Модель: {model_path}")


Python 3.13.6; NumPy 2.4.3
Модель: D:\Аспа\Kalmykov_PhD\Colab Notebooks\two_layer_model.py


In [2]:
# Синтетическая рабочая точка; не параметры добровольца/пациента.
RHO1 = 5.0
RHO2 = 20.0
H = 0.020
BETA = 0.5
MODEL_VALIDITY = "unverified_without_subject_specific_CT_FEM_Lmax"
OBSERVATION_OPERATOR = "signed_real_Z; experimental_magnitude_not_yet_verified"

SIZE_SETS_MM = {
    "9 размеров; поздняя копия 100 мм исключена": (50, 60, 70, 80, 90, 110, 120, 130, 140),
    "10 размеров; 90 и 100 мм независимы до QC": (50, 60, 70, 80, 90, 100, 110, 120, 130, 140),
}

print("status=illustrative_unvalidated")
print("reduced_method_status=not_validated_against_CT")
print(f"model_validity={MODEL_VALIDITY}")
print(f"observation_operator={OBSERVATION_OPERATOR}")
print(f"rho1={RHO1:g} Ом·м; rho2={RHO2:g} Ом·м; h={H*1000:g} мм; beta={BETA:g}")


status=illustrative_unvalidated
reduced_method_status=not_validated_against_CT
model_validity=unverified_without_subject_specific_CT_FEM_Lmax
observation_operator=signed_real_Z; experimental_magnitude_not_yet_verified
rho1=5 Ом·м; rho2=20 Ом·м; h=20 мм; beta=0.5


In [3]:
def log_jacobian(sizes_m, rho1=RHO1, rho2=RHO2, h=H):
    rows = []
    for size_m in np.asarray(sizes_m, dtype=float):
        a, b = tlm.geometry_from_size(float(size_m), BETA)
        result = tlm.evaluate(rho1, rho2, h, a, b)
        rows.append([
            rho1 * result.d_rho1 / result.z,
            rho2 * result.d_rho2 / result.z,
            h * result.d_h / result.z,
        ])
    matrix = np.asarray(rows, dtype=float)
    if not np.isfinite(matrix).all():
        raise RuntimeError("non-finite logarithmic Jacobian")
    return matrix


def subset_metrics(sizes_mm):
    sizes_m = np.asarray(sizes_mm, dtype=float) / 1000.0
    matrix = log_jacobian(sizes_m)
    singular = np.linalg.svd(matrix, compute_uv=False)
    rank = int(np.linalg.matrix_rank(matrix))
    result = {
        "sizes_mm": tuple(int(value) for value in sizes_mm),
        "n": len(sizes_mm),
        "rank": rank,
        "sigma_max": float(singular[0]),
        "sigma_min": float(singular[-1]),
        "condition": float(singular[0] / singular[-1]) if singular[-1] > 0 else np.inf,
        "unit_var_log_rho2": np.inf,
        "unit_var_log_h": np.inf,
    }
    if rank == 3:
        covariance = np.linalg.inv(matrix.T @ matrix)
        result["unit_var_log_rho2"] = float(covariance[1, 1])
        result["unit_var_log_h"] = float(covariance[2, 2])
    return result


def enumerate_subsets(sizes_mm, minimum_size=3):
    table = []
    for count in range(minimum_size, len(sizes_mm) + 1):
        table.extend(subset_metrics(subset) for subset in combinations(sizes_mm, count))
    return table


def best_by_cardinality(table, key, reverse=False):
    best = {}
    for row in table:
        current = best.get(row["n"])
        if current is None:
            best[row["n"]] = row
        elif reverse and row[key] > current[key]:
            best[row["n"]] = row
        elif not reverse and row[key] < current[key]:
            best[row["n"]] = row
    return best


# Точные структурные самопроверки.
pair = log_jacobian([0.050, 0.140])
assert pair.shape == (2, 3) and np.linalg.matrix_rank(pair) == 2
triple = log_jacobian([0.050, 0.090, 0.140])
assert triple.shape == (3, 3) and np.linalg.matrix_rank(triple) == 3

all_sizes = np.asarray(SIZE_SETS_MM[list(SIZE_SETS_MM)[1]], dtype=float) / 1000.0
with_gain = np.column_stack([log_jacobian(all_sizes), np.ones(len(all_sizes))])
scale_null = np.array([1.0, 1.0, 0.0, -1.0])
np.testing.assert_allclose(with_gain @ scale_null, 0.0, rtol=0.0, atol=1e-10)
assert np.linalg.matrix_rank(with_gain) == 3

tables = {name: enumerate_subsets(sizes) for name, sizes in SIZE_SETS_MM.items()}
assert len(tables[list(tables)[0]]) == 466
assert len(tables[list(tables)[1]]) == 968
print("Самопроверки ранга, gain-вырождения и полного перебора: пройдены")


Самопроверки ранга, gain-вырождения и полного перебора: пройдены


In [4]:
for set_name, table in tables.items():
    print(f"\n{set_name}; наборов мощности >=3: {len(table)}")
    by_e_opt = best_by_cardinality(table, "sigma_min", reverse=True)
    by_target = best_by_cardinality(table, "unit_var_log_rho2", reverse=False)
    print("n | max sigma_min: размеры; cond | min unit_var_ln_rho2: размеры; unit_var_ln_h")
    for count in sorted(by_e_opt):
        e_row = by_e_opt[count]
        t_row = by_target[count]
        print(
            f"{count:1d} | {e_row['sigma_min']:.6f}: {e_row['sizes_mm']}; {e_row['condition']:.1f} | "
            f"{t_row['unit_var_log_rho2']:.2f}: {t_row['sizes_mm']}; {t_row['unit_var_log_h']:.2f}"
        )

    full_row = subset_metrics(SIZE_SETS_MM[set_name])
    print(
        "Все доступные размеры при Sigma=I: "
        f"sigma_min={full_row['sigma_min']:.6f}; cond={full_row['condition']:.1f}; "
        f"unit_var_ln_rho2={full_row['unit_var_log_rho2']:.2f}; "
        f"unit_var_ln_h={full_row['unit_var_log_h']:.2f}"
    )



9 размеров; поздняя копия 100 мм исключена; наборов мощности >=3: 466
n | max sigma_min: размеры; cond | min unit_var_ln_rho2: размеры; unit_var_ln_h
3 | 0.042711: (50, 90, 140); 37.8 | 345.39: (50, 80, 140); 209.65
4 | 0.051712: (50, 80, 90, 140); 36.4 | 239.41: (50, 80, 90, 140); 135.31
5 | 0.054018: (50, 70, 80, 90, 140); 39.1 | 216.84: (50, 70, 80, 90, 140); 126.23
6 | 0.055615: (50, 70, 80, 90, 130, 140); 41.4 | 197.57: (50, 70, 80, 90, 130, 140); 123.71
7 | 0.056903: (50, 70, 80, 90, 110, 130, 140); 43.8 | 191.28: (50, 70, 80, 90, 110, 130, 140); 115.65
8 | 0.056996: (50, 60, 70, 80, 90, 110, 130, 140); 46.8 | 190.93: (50, 70, 80, 90, 110, 120, 130, 140); 115.45
9 | 0.056998: (50, 60, 70, 80, 90, 110, 120, 130, 140); 49.5 | 190.89: (50, 60, 70, 80, 90, 110, 120, 130, 140); 114.00
Все доступные размеры при Sigma=I: sigma_min=0.056998; cond=49.5; unit_var_ln_rho2=190.89; unit_var_ln_h=114.00

10 размеров; 90 и 100 мм независимы до QC; наборов мощности >=3: 968
n | max sigma_min: р

In [5]:
def solve_positive_signed_subset(sizes_m, observations, initial=(5.0, 20.0, 0.020), max_iter=40):
    sizes_m = np.asarray(sizes_m, dtype=float)
    observations = np.asarray(observations, dtype=float)
    if len(sizes_m) < 3 or observations.shape != sizes_m.shape or np.any(observations <= 0):
        raise ValueError("at least three positive signed-Z observations are required")

    log_parameters = np.log(np.asarray(initial, dtype=float))
    for _ in range(max_iter):
        rho1, rho2, h = np.exp(log_parameters)
        predictions = []
        for size_m in sizes_m:
            a, b = tlm.geometry_from_size(float(size_m), BETA)
            predictions.append(tlm.transfer_impedance(rho1, rho2, h, a, b))
        predictions = np.asarray(predictions)
        residual = np.log(predictions) - np.log(observations)
        if np.linalg.norm(residual, ord=np.inf) < 1e-12:
            return rho1, rho2, h
        step, *_ = np.linalg.lstsq(log_jacobian(sizes_m, rho1, rho2, h), residual, rcond=None)
        log_parameters -= step
    raise RuntimeError("subset Newton solver did not converge")


# Только algebra/code self-test на данных той же модели.
true_parameters = (6.0, 18.0, 0.025)
fit_sizes = np.array([0.050, 0.090, 0.140])
fit_observations = []
for size_m in fit_sizes:
    a, b = tlm.geometry_from_size(float(size_m), BETA)
    fit_observations.append(tlm.transfer_impedance(*true_parameters, a, b))
recovered = solve_positive_signed_subset(fit_sizes, fit_observations)
np.testing.assert_allclose(recovered, true_parameters, rtol=1e-9, atol=1e-10)

holdout_sizes = np.array([0.060, 0.110, 0.130])
holdout_residuals = []
for size_m in holdout_sizes:
    a, b = tlm.geometry_from_size(float(size_m), BETA)
    truth = tlm.transfer_impedance(*true_parameters, a, b)
    prediction = tlm.transfer_impedance(*recovered, a, b)
    holdout_residuals.append(prediction - truth)
np.testing.assert_allclose(holdout_residuals, 0.0, rtol=0.0, atol=1e-9)
print("Синтетический self-test трёхпараметрического обращения и holdout: пройден")


Синтетический self-test трёхпараметрического обращения и holdout: пройден


## Интерпретация

Перебор показывает только локальную геометрию идеальной модели в одной
синтетической точке. Для каждой мощности можно назвать набор с наибольшим
$\sigma_{min}$ или наименьшей условной дисперсией при $\Sigma=I$, но эти
оптимумы могут различаться. Их нельзя превращать в протокол без принятого
целевого параметра и цены дополнительной записи.

При условии независимых одинаковых ошибок полный набор формально накапливает
наибольшую информацию. В реальном эксперименте это условие особенно слабое:
все размеры записаны последовательно в одном порядке, а повторов размера нет.
Добавление строки может добавлять не независимую информацию, а ещё одно
проявление временного тренда, переклейки или систематики сборки.

Трёхточечный synthetic inversion/holdout проверяет только код. Он не
доказывает глобальную единственность, устойчивость к ошибкам или соответствие
расчётной $h$ анатомии.


## Программа доказательства сокращённой постановки

1. На добровольцах выполнить полную задачу с $h_{CT}$ (`32.02`) и получить
   референсные оценки/профили.
2. Для тех же принятых данных решить задачу с неизвестной $h$, не используя КТ
   при оценке, затем сравнить $\widehat h$ с $h_{CT}$ и перенос ошибок на
   $\rho_2$.
3. Повторить по субъектам, дыхательным состояниям и приборам раздельно; общие
   $h$/$\rho_1$ между состояниями проверять, а не предполагать по умолчанию.
4. Оценить gain/offset и реальную ковариацию. При неизвестном gain абсолютные
   $\rho$ неидентифицируемы независимо от числа размеров.
5. Использовать профили функции потерь, multi-start, holdout-размеры,
   leave-one-size-out и отказ, если минимум не внутренний/устойчивый.
6. Задать заранее допустимую ошибку $h$ и $\rho_2$, стоимость записи и
   субъектное допустимое множество по `20.12`.
7. Только после этого выбрать минимальный набор и проверить его на новом
   добровольце/независимой сессии до перехода к пациентам без КТ.

Текущий итог:

```text
reduced_unknown_h_status = not_validated_against_CT
minimal_set_status = not_determined
reason = calibration_covariance_accuracy_cost_and_external_validation_missing
```

Это завершает структурную серию `32`, но не завершает выбор реальных сборок.
